Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2025/26 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Homework Template

**Important:** All data in the virtual machine is lost, once the virtual machine is deleted.

So, please make sure that you copy this template and the ground truth data into your `mnt/home/...` folder to allow for persistent storage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import tensorflow as tf
from tensorflow import keras

from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# we can use PyTorch or TensorFlow with Keras
torch.__version__, tf.__version__, keras.__version__

In [ ]:
# we might want to use double precision
# in PyTorch and TensorFlow because
# the ground truth data is dtype=float64
torch.set_default_dtype(torch.float64)
tf.keras.backend.set_floatx('float64')

# Data

In [ ]:
# load provided ground truth data
X = np.load('ground_truth_data_x.npy')
Y = np.load('ground_truth_data_y.npy')
Y = Y[:, None]  # make it a 2D numpy array, i.e. a column vector
N, F = X.shape[0], X.shape[1]

print('data samples N:', N, '\nfeatures F:', F)
print(X.dtype, Y.dtype, X.shape, Y.shape)

# make sure X is full column rank -> then X has a left-inverse
print('X is full column rank:', np.linalg.matrix_rank(X) == F)

In [ ]:
# plot some data
plt.plot(np.arange(N), Y)
plt.xlabel('data sample index k')
plt.ylabel('y[k]')
plt.title('ground truth data Y')
plt.xlim([0, N-1])
plt.ylim([-0.4, +0.2])
plt.grid(True)
plt.tight_layout()

# Linear Regression / Ordinary Least Squares

We might want to use a linear model, i.e. **linear regression** / ordinary least squares (OLS).

We have dealt with OLS in exercise 5 [Line Fit with Linear Regression](../line_fit_linear_regression.ipynb) and in the lecture (`univariate_linear_regression_demo.ipynb` and `bivariate_linear_regression_demo.ipynb`)

## Matrix Inverse Model for OLS

The code below will do OLS using the closed form solution of linear algebra fundamentals.

In [ ]:
# fit / train the model with full data set == full batch
# note that we intentionally overfit the model in this homework task
X_left_inverse = np.linalg.inv(X.T @ X) @ X.T
w = X_left_inverse @ Y  # get model weights w by solving an inverse problem
print('model parameters:\n', w)
# predict with the model / make inference
Y_predict = X @ w  # forward propagation, often also denoted y_hat
# for convenience of this homework task we use train data == test data
# please, never ever do this for practical applications!

# get error / residual e, get loss L & get empirical risk ER
e = Y - Y_predict
L = e.T @ e  # =||e||^2
ER = L / N
print('empirical risk:', ER[0, 0])  # 0.00342780054087372 = 3.42780054087372e-3
# can we achieve lower ER with non-linear models?!
#
# in fact the ground truth data y=f(x) originates from a non-linear function f,
# hence a linear model somehow fails to do a optimum prediction job

The model performance is not really convincing, because the data originates from a non-linear model.

So, a linear model is probably not the best choice.

Hence, we need to go for a non-linear model.

This motivates the homework task.

# PyTorch Model for OLS

A linear model designed, compiled and trained with PyTorch could go as follows.

Please see the other Torch models
- [binary_logistic_regression_torch.ipynb](../binary_logistic_regression_torch.ipynb)
- [binary_logistic_regression_torch_with_hidden_layers.ipynb](../binary_logistic_regression_torch_with_hidden_layers.ipynb)
for further inspiration, how the homework task might be implemented.

In [ ]:
# these parameter choices yield the same results
# as the matrix inversion model above
batch_size = N  # full batch
num_epochs = 15
learning_rate = 0.5

In [ ]:
# for convenience of this homework task we use train data == test data
# please, never ever do this for practical applications!
data_train = TensorDataset(torch.DoubleTensor(X),
                           torch.DoubleTensor(Y))
data_test = TensorDataset(torch.DoubleTensor(X),
                          torch.DoubleTensor(Y))
data_train_loader = DataLoader(dataset=data_train,
                               batch_size=batch_size,
                               shuffle=False)

In [ ]:
class Model(torch.nn.Module):

    def __init__(self):
        super(Model, self).__init__()

        self.linear = torch.nn.Linear(2, 1, bias=False)

    def forward(self, x):
        x = self.linear(x)
        return x


model = Model()

empirical_risk = torch.nn.MSELoss(reduction='mean')  # regression uses MSEloss
# for convenience we use SGD with defaults == plain GD
optimizer = torch.optim.SGD(
    model.parameters(), lr=learning_rate)

model = torch.compile(model)

In [ ]:
for epoch in range(num_epochs):
    for i, batch in enumerate(data_train_loader, 1):
        X_batch, Y_batch = batch[0], batch[1]
        Y_batch_predict = model(X_batch)
        loss = empirical_risk(Y_batch_predict, Y_batch)
        loss.backward()  # back prop
        optimizer.step()  # gradient descent
        optimizer.zero_grad()  # reset gradients for next iteration
with torch.no_grad():
    loss = empirical_risk(
        model(data_test[:][0]),  # predicted Y
        data_test[:][1])  # ground truth Y
    # PyTorch model:
    print('PyTorch model:\t\t loss = %0.15e' % loss)
    # closed form model
    print('Matrix inverse model:\t loss = %0.15e' % ER[0, 0])

In [ ]:
w_torch = model.linear.weight.detach().numpy()
print('PyTorch model parameters:\t',
      '%0.15f' % w_torch[0, 0], '%0.15f' % w_torch[0, 1])
print('Matrix inverse model parameters:',
      '%0.15f' % w[0, 0], '%0.15f' % w[1, 0])

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- feel free to use the notebooks for your own purposes
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.